# SOLANGE — Generate the ARID2 JW Hamiltonian (Jupyter)

Runs `generate_expansion_jw.py` to compute the real PySCF CASSCF(2,2)/STO-3G + Jordan-Wigner
Hamiltonian for **ARID2** (the only new gene) and append it to `backend/jw_hamiltonians.json`.
It **skips** the 30 genes already present, so it computes only ARID2.

**Before running:** put the two files I sent in the SAME folder as this notebook —
`generate_expansion_jw.py` and `jw_hamiltonians.json`.
(If you have the full repo cloned, just open this notebook at the repo root — everything is already in place;
run `git pull` first to be on the latest version.)

Then run the cells top to bottom.

In [ ]:
# 1) Install dependencies into THIS kernel's environment
import sys
!{sys.executable} -m pip install -q pyscf openfermion scipy numpy
print('dependencies installed')

In [ ]:
# 2) Locate / prepare the files, and confirm we are on the LATEST script (with ARID2)
import os, json

assert os.path.exists('generate_expansion_jw.py'), \
    'Put generate_expansion_jw.py in this folder (the latest one I sent).'

# the script reads/writes backend/jw_hamiltonians.json next to itself
os.makedirs('backend', exist_ok=True)
if not os.path.exists('backend/jw_hamiltonians.json'):
    if os.path.exists('jw_hamiltonians.json'):
        os.replace('jw_hamiltonians.json', 'backend/jw_hamiltonians.json')
        print('moved jw_hamiltonians.json -> backend/')
    else:
        raise FileNotFoundError('Need jw_hamiltonians.json (here) or backend/jw_hamiltonians.json')

script = open('generate_expansion_jw.py', encoding='utf-8').read()
assert 'ARID2_LOF' in script, \
    'This generate_expansion_jw.py is OLD (no ARID2). Use the latest file I sent, or git pull.'

d = json.load(open('backend/jw_hamiltonians.json'))
print('current genes in JSON:', len(d))
print('ARID2_LOF already present:', 'ARID2_LOF' in d)
print('OK — latest script has ARID2, ready to run.')

In [ ]:
# 3) Run the generator (uses this kernel's Python, so pyscf is available)
import subprocess, sys
r = subprocess.run([sys.executable, 'generate_expansion_jw.py'],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print('--- STDERR ---')
    print(r.stderr[-2500:])

In [ ]:
# 4) Verify ARID2_LOF was added correctly
import json
d = json.load(open('backend/jw_hamiltonians.json'))
print('genes now:', len(d))
if 'ARID2_LOF' in d:
    e = d['ARID2_LOF']
    for side in ('native', 'mutant'):
        s = e[side]
        print(f"  ARID2_LOF/{side}: compound={s['compound']:12s} "
              f"n_paulis={s['n_paulis']}  e_casscf={s['e_casscf']:.6f} Ha")
    print('\n✅ SUCCESS — ARID2_LOF added. Send backend/jw_hamiltonians.json back to me (Claude),')
    print('   or commit it:  git add backend/jw_hamiltonians.json && git commit -m "Add ARID2 JW Hamiltonian" && git push')
else:
    print('\n❌ ARID2_LOF NOT added — check the run output in cell 3 above.')

## After a successful run

The file `backend/jw_hamiltonians.json` now contains **31 genes** (30 + `ARID2_LOF`).

1. **Send it back to me** and I will sync (root + backend), verify, and push — ARID2 becomes a live target; **or**
2. If you cloned the repo, just commit and push it yourself (command printed in cell 4).

A CASSCF `WARNING: did not fully converge` line is harmless — the JW terms are still valid.